# General

For more informations, see the documentation *LLM and GenAI*.

# Import & Configs

In [7]:
import json
import pandas as pd

In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [9]:
%load_ext autoreload
%autoreload 2

from src.retrieval.retriever import MedicalRetriever
from src.pipeline.medical_assistant import ask_medical_assistant

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Questions Set

In [10]:
with open(
    "../data/evaluations/gold_questions.json"
) as f:

    questions = json.load(f)

questions

[{'question': 'What is glioblastoma?'},
 {'question': 'How is glioblastoma prognosis evaluated?'},
 {'question': 'What MRI techniques are used for glioblastoma?'},
 {'question': 'What is peritumoral edema?'},
 {'question': 'What is pseudoprogression?'},
 {'question': 'How is tumor progression detected?'},
 {'question': 'What biomarkers are associated with glioblastoma?'},
 {'question': 'What are the limitations of MRI in glioma diagnosis?'},
 {'question': 'How is overall survival measured?'},
 {'question': 'What treatments are commonly used for glioblastoma?'}]

# Retrieval Evaluation

In [11]:
retriever = MedicalRetriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
results_retriever = []

for item in questions:
    
    question = item["question"]
    print(f'{question:-^80}')
    #output = retriever.retrieve(question)
    output = retriever.retrieve_diverse_articles(question)
    results_retriever.append(output)
    print('\n')
    for key, val in output.items():
        #print(f'--- {key}:')
        if key == 'documents':
            for i, doc in enumerate(val[0]):
                print(f'[{output['ids'][0][i]}]\n{doc}\n')
        else:
            #print(f'{val}')
            pass
        #print('\n')
    print("\n\n")

-----------------------------What is glioblastoma?------------------------------
Original query: What is glioblastoma?
Processed query: glioblastoma


[39513861_0]
The Three Pillars of Glioblastoma: A Systematic Review and Novel Analysis of Multi-Omics and Clinical Data. Glioblastoma is the most fatal and common malignant brain tumor, excluding metastasis and with a median survival of approximately one year. While solid tumors benefit from newly approved drugs, immunotherapy, and prevention, none of these scenarios are opening for glioblastoma.

[40542948_0]
Unlocking glioblastoma: breakthroughs in molecular mechanisms and next-generation therapies. Glioblastoma (GB) remains the most aggressive primary brain tumor in adults, characterized by rapid progression, recurrence, and resistance to conventional therapies. Despite advancements in surgical resection, radiation, and chemotherapy, long-term survival rates remain low.

[41777607_0]
Neurosurgical and neuro-oncological outcomes of con

### V1

| Question  | Retrieval | Comments |
|-----|:-----:|-----|
| Q1 | 4 |  |
| Q2 | 3 |  |
| Q3 | 2 |  |
| Q4 | 3 | Confusion btw glioma & glioblastoma |
| Q5 | 1 | Not Relevant | 
| Q6 | 2 | Relevance ? | 
| Q7 | 3 |  |
| Q8 | 2 | Only doc 1 is relevant |
| Q9 | 2 | Doc 1 : not relevant |
| Q10 | 1 | Confusion btw glioma & glioblastoma |

With:
- 1 = bad
- 5 = excellent

**Definitions:**
- **Retrieval:** Have the correct documents been found?

Identified Limitations:
- Chunks that are too specialized.
- Chunks at the wrong level
- Diversification issues

### V2

| Question  | Retrieval | Comments |
|-----|:-----:|-----|
| Q1 | 4 | 1 doc too large, others good |
| Q2 | 3 | Correct, but not ideal |
| Q3 | 4 |  |
| Q4 | 4 |  |
| Q5 | 3 | Doc 2 : weak | 
| Q6 | 2 | progression more than detection | 
| Q7 | 4 |  |
| Q8 | 2 | Only doc 1 is relevant |
| Q9 | 1 | Not relevant |
| Q10 | 2 | Standard treatments not found |

With:
- 1 = bad
- 5 = excellent

**Definitions:**
- **Retrieval:** Have the correct documents been found?

## V3

In [ ]:
[5,4,5,4,5,2,5,3,2,3]
[
    "Two excellent definitional reviews in the top spots.",
    "Relevant documents on survival, biomarkers, and prognostic prediction. Not perfect, but sufficient.",
    "Excellent MRI documents, including a dedicated review.",
    "The literature clearly defines peritumoral edema even outside the context of glioblastoma.",
    "The best available evidence comes to light immediately (SR + meta-analysis).",
    "The documents focus primarily on tumor progression, not on its detection.",
    "Excellent summary, with a systematic review focused on biomarkers.",
    "A relevant review, but the other documents are not very relevant to the question.",
    "Documents that mention survival measures but do not provide a direct explanation from the OS.",
    "Treatments are available, but mainly in the context of recurrence or disease progression. They aren't really the standard treatments for GBM."
]

| Question  | Retrieval | Comments |
|-----|:-----:|-----|
| Q1 | 5 | Two excellent definitional reviews in the top spots. |
| Q2 | 4 | Relevant documents on survival, biomarkers, and prognostic prediction. Not perfect, but sufficient. |
| Q3 | 5 | Excellent MRI documents, including a dedicated review. |
| Q4 | 4 | The literature clearly defines peritumoral edema even outside the context of glioblastoma. |
| Q5 | 5 | The best available evidence comes to light immediately (SR + meta-analysis). | 
| Q6 | 2 | The documents focus primarily on tumor progression, not on its detection. | 
| Q7 | 5 | Excellent summary, with a systematic review focused on biomarkers. |
| Q8 | 3 | A relevant review, but the other documents are not very relevant to the question. |
| Q9 | 2 | Documents that mention survival measures but do not provide a direct explanation from the OS. |
| Q10 | 3 | Treatments are available, but mainly in the context of recurrence or disease progression. They aren't really the standard treatments for GBM. |

With:
- 1 = bad
- 5 = excellent

**Definitions:**
- **Retrieval:** Have the correct documents been found?

# Answer Evaluation

In [13]:
results_answer = []

for item in questions:
    
    question = item["question"]
    print(f'{question:-^80}')

    output = ask_medical_assistant(
        question,
        retriever,
        #debug=True
    )

    results_answer.append(output)
    print("\n\n")

-----------------------------What is glioblastoma?------------------------------
Original query: What is glioblastoma?
Processed query: glioblastoma
Loaded 3307 articles.
Glioblastoma is the most fatal and common malignant brain tumor, excluding metastasis and with a median survival of approximately one year. It is characterized by rapid progression, recurrence, and resistance to conventional therapies. Despite advancements in surgical resection, radiation, and chemotherapy, long-term survival rates remain low.
Sources used:
[1] The Three Pillars of Glioblastoma: A Systematic Review and Novel Analysis of Multi-Omics and Clinical Data (2024) PMID:39513861
[2] Unlocking glioblastoma: breakthroughs in molecular mechanisms and next-generation therapies (2025) PMID:40542948
[3] Neurosurgical and neuro-oncological outcomes of confirmatory brain biopsies in patients with glioblastoma: a real-life monocentric experience (2026) PMID:41777607




--------------------How is glioblastoma prognosis

### V1

| Question  | Relevance | Faithfulness | Clarity |
|:-----|:-----:|:-----:|:-----:|
| Q1 | 5 | 5 | 5 |
| Q2 | 4 | 4 | 3 |
| Q3 | 2 | 4 | 4 |
| Q4 | 5 | 5 | 4 |
| Q5 |  4 | 4 | 4 |
| Q6 |  3 | 3 | 4 |
| Q7 |  2 | 4 | 4 |
| Q8 |  5 | 5 | 4 |
| Q9 |  1 | 4 | 3 |
| Q10 |  1 | 5 | 2 |

With:
- 1 = bad
- 5 = excellent



**Definitions:**
- **Relevance:** Does it answer the question?
- **Accuracy:** Is it accurate in the context?
- **Clarity:** Is it understandable?


Identified limitations:
- Duplicate documents
- Overly simplistic retrieval
- Insufficient model

### V2

| Question  | Relevance | Faithfulness | Clarity | Comments |
|:-----|:-----:|:-----:|:-----:|:-----|
| Q1 | 5 | 4 | 5 | “10–20% of all primary brain tumors” seems far-fetched. |
| Q2 | 3 | 5 | 4 | Answer too specific (hematology) |
| Q3 | 3 | 4 | 4 | DWI/IVIM/DSC are explicitly in the excerpts provided ? |
| Q4 | 5 | 5 | 5 |  |
| Q5 |  4 | 5 | 5 |  |
| Q6 |  3 | 5 | 4 | “how progression patterns are studied” rather than “how progression is detected.” |
| Q7 |  3 | 5 | 5 | response too restrictive regarding biomarkers |
| Q8 |  4 | 4 | 4 | Some limitations are well-founded, while others seem to be extrapolated. |
| Q9 |  1 | 5 | 3 | off-topic |
| Q10 |  1 | 5 | 1 | “I don't know” makes sense in the context but does not answer the user's question. |

With:
- 1 = bad
- 5 = excellent



**Definitions:**
- **Relevance:** Does it answer the question?
- **Accuracy:** Is it accurate in the context?
- **Clarity:** Is it understandable?

## V3

| Question  | Relevance | Faithfulness | Clarity | Comments |
|:-----|:-----:|:-----:|:-----:|:-----|
| Q1 | 5 | 5 | 5 | A short, accurate answer directly supported by the sources. |
| Q2 | 3 | 4 | 4 | Partially answers the question. The model focuses more on observed survival than on prognostic assessment. |
| Q3 | 5 | 5 | 5 | A direct and comprehensive answer. |
| Q4 | 5 | 5 | 5 | A clear and accurate definition. |
| Q5 |  5 | 5 | 5 | A very good definition of pseudoprogression. |
| Q6 |  2 | 4 | 3 | Off-topic response: factors influencing progression rather than detection methods. |
| Q7 |  3 | 5 | 4 | Correct, but too restrictive: only one biomarker is mentioned. |
| Q8 |  4 | 3 | 4 | A plausible answer, but several limitations seem to have been extrapolated from the context. |
| Q9 |  4 | 5 | 4 | Correctly answer the question using the OS's statistical measurement methods. |
| Q10 |  1 | 5 | 4 | The model implicitly acknowledges the lack of context. The response is accurate but does not answer the question. |

With:
- 1 = bad
- 5 = excellent



**Definitions:**
- **Relevance:** Does it answer the question?
- **Accuracy:** Is it accurate in the context?
- **Clarity:** Is it understandable?

# Dowload Results

In [23]:
rows = []

for question, retrieval, answer in zip(
    questions,
    results_retriever,
    results_answer
):

    docs = retrieval["documents"][0]
    metas = retrieval["metadatas"][0]
    distances = retrieval["distances"][0]

    rows.append({
        "question": question["question"],

        "answer": answer,

        "doc_1": docs[0],
        "doc_2": docs[1],
        "doc_3": docs[2],

        "pmid_1": metas[0]["pmid"],
        "pmid_2": metas[1]["pmid"],
        "pmid_3": metas[2]["pmid"],

        "distance_1": distances[0],
        "distance_2": distances[1],
        "distance_3": distances[2],
    })

df_eval = pd.DataFrame(rows)
df_eval.head()

,question,answer,doc_1,doc_2,doc_3,pmid_1,pmid_2,pmid_3,distance_1,distance_2,distance_3
0,What is glioblastoma?,"{'results': {'ids': [['39513861_0', '40542948_...",The Three Pillars of Glioblastoma: A Systemati...,Unlocking glioblastoma: breakthroughs in molec...,Neurosurgical and neuro-oncological outcomes o...,39513861,40542948,41777607,0.480071,0.492108,0.506533
1,How is glioblastoma prognosis evaluated?,"{'results': {'ids': [['41091233_5', '39731064_...",Conditional 5-year survival after surviving on...,This systematic review summarizes the current ...,Molecular Biomarkers in Glioblastoma: A System...,41091233,39731064,36012105,0.359277,0.423970,0.428856
2,What MRI techniques are used for glioblastoma?,"{'results': {'ids': [['41749898_0', '42162948_...",Exploring the Role of Advanced MRI in Understa...,Impact on survival of glioblastoma patient's i...,"We aimed to show the safety of a small-margin,...",41749898,42162948,42134380,0.467751,0.532589,0.537395
3,What is peritumoral edema?,"{'results': {'ids': [['41740788_1', '42217213_...","(2) For 2 patients, we took the following meas...","Notwithstanding its benign classification, SM ...","All tumors showed significant enhancement, wit...",41740788,42217213,38254159,0.596067,0.755063,0.780171
4,What is pseudoprogression?,"{'results': {'ids': [['38992848_0', '40768078_...",Perioperative imaging predictors of tumor prog...,Artificial intelligence algorithms for differe...,Additional subclassification was performed bas...,38992848,40768078,40360047,1.095843,1.136668,1.234785


In [24]:
df_eval["retrieval"] = None
df_eval["retrieval_comments"] = ""

df_eval["relevance"] = None
df_eval["faithfulness"] = None
df_eval["clarity"] = None

In [25]:
df_eval["retrieval"] = [5,4,5,4,5,2,5,3,2,3]
df_eval["retrieval_comments"] = [
    "Two excellent definitional reviews in the top spots.",
    "Relevant documents on survival, biomarkers, and prognostic prediction. Not perfect, but sufficient.",
    "Excellent MRI documents, including a dedicated review.",
    "The literature clearly defines peritumoral edema even outside the context of glioblastoma.",
    "The best available evidence comes to light immediately (SR + meta-analysis).",
    "The documents focus primarily on tumor progression, not on its detection.",
    "Excellent summary, with a systematic review focused on biomarkers.",
    "A relevant review, but the other documents are not very relevant to the question.",
    "Documents that mention survival measures but do not provide a direct explanation from the OS.",
    "Treatments are available, but mainly in the context of recurrence or disease progression. They aren't really the standard treatments for GBM."
]

In [26]:
df_eval["relevance"] = [5,3,5,5,5,2,3,4,4,1]
df_eval["faithfulness"] = [5,4,5,5,5,4,5,3,5,5]
df_eval["clarity"] = [5,4,5,5,5,3,4,4,4,4]
df_eval["answer_comments"] = [
    "A short, accurate answer directly supported by the sources.",
    "Partially answers the question. The model focuses more on observed survival than on prognostic assessment.",
    "A direct and comprehensive answer.",
    "A clear and accurate definition.",
    "A very good definition of pseudoprogression.",
    "Off-topic response: factors influencing progression rather than detection methods.",
    "Correct, but too restrictive: only one biomarker is mentioned.",
    "A plausible answer, but several limitations seem to have been extrapolated from the context.",
    "Correctly answer the question using the OS's statistical measurement methods.",
    "The model implicitly acknowledges the lack of context. The response is accurate but does not answer the question."
]

In [27]:
df_eval.head()

,question,answer,doc_1,doc_2,doc_3,pmid_1,pmid_2,pmid_3,distance_1,distance_2,distance_3,retrieval,retrieval_comments,relevance,faithfulness,clarity,answer_comments
0,What is glioblastoma?,"{'results': {'ids': [['39513861_0', '40542948_...",The Three Pillars of Glioblastoma: A Systemati...,Unlocking glioblastoma: breakthroughs in molec...,Neurosurgical and neuro-oncological outcomes o...,39513861,40542948,41777607,0.480071,0.492108,0.506533,5,Two excellent definitional reviews in the top ...,5,5,5,"A short, accurate answer directly supported by..."
1,How is glioblastoma prognosis evaluated?,"{'results': {'ids': [['41091233_5', '39731064_...",Conditional 5-year survival after surviving on...,This systematic review summarizes the current ...,Molecular Biomarkers in Glioblastoma: A System...,41091233,39731064,36012105,0.359277,0.423970,0.428856,4,"Relevant documents on survival, biomarkers, an...",3,4,4,Partially answers the question. The model focu...
2,What MRI techniques are used for glioblastoma?,"{'results': {'ids': [['41749898_0', '42162948_...",Exploring the Role of Advanced MRI in Understa...,Impact on survival of glioblastoma patient's i...,"We aimed to show the safety of a small-margin,...",41749898,42162948,42134380,0.467751,0.532589,0.537395,5,"Excellent MRI documents, including a dedicated...",5,5,5,A direct and comprehensive answer.
3,What is peritumoral edema?,"{'results': {'ids': [['41740788_1', '42217213_...","(2) For 2 patients, we took the following meas...","Notwithstanding its benign classification, SM ...","All tumors showed significant enhancement, wit...",41740788,42217213,38254159,0.596067,0.755063,0.780171,4,The literature clearly defines peritumoral ede...,5,5,5,A clear and accurate definition.
4,What is pseudoprogression?,"{'results': {'ids': [['38992848_0', '40768078_...",Perioperative imaging predictors of tumor prog...,Artificial intelligence algorithms for differe...,Additional subclassification was performed bas...,38992848,40768078,40360047,1.095843,1.136668,1.234785,5,The best available evidence comes to light imm...,5,5,5,A very good definition of pseudoprogression.


In [28]:
version = 3
model = "qwen2.5:1.5b"
step = "corpus_improvement"

df_eval.to_csv(
    f"../data/evaluations/V{version}_{step}_evalN7_retriev_&_answ_model-{model}.csv",
    index=False
)

___

In [29]:
summary = {
    "retrieval_mean": df_eval["retrieval"].mean(),
    "relevance_mean": df_eval["relevance"].mean(),
    "faithfulness_mean": df_eval["faithfulness"].mean(),
    "clarity_mean": df_eval["clarity"].mean(),
}

In [30]:
print(summary)

{'retrieval_mean': np.float64(3.8), 'relevance_mean': np.float64(3.7), 'faithfulness_mean': np.float64(4.6), 'clarity_mean': np.float64(4.3)}


### V1

Means:

| Retrieval  | Relevance | Faithfulness | Clarity |
|:-----|:-----:|:-----:|:-----:|
| 2.3 | 3.2 | 4.3 | 3.7 |

### V2

Means:

| Retrieval  | Relevance | Faithfulness | Clarity |
|:-----|:-----:|:-----:|:-----:|
| 2.9 | 3.2 | 4.7 | 4.0 |

### Save

In [22]:
df_eval.to_csv(
    f"../data/evaluations/V{version}_{step}_evalN7_summary_model-{model}.csv",
    index=False
)